# Credit Risk — Cleaning (NO encoding)
This notebook handles missing values, outliers, and adds `customer_id`, then saves the clean data. **Categories stay as words — no encoding here.**

### Import tools and load the data

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt

file_path = './credit_risk_dataset.csv'
df = pd.read_csv(file_path)

### First look
See the rows, size, where nulls are, and the number summary.

In [ ]:
print(df.head())
print('Shape:', df.shape)
print(df.isnull().sum())
print(df.describe())

### Step 1 — Handle missing values
`person_emp_length` → median. `loan_int_rate` → median of its own loan grade (rate depends on grade).

In [ ]:
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())

df['loan_int_rate'] = df.groupby('loan_grade')['loan_int_rate'].transform(
    lambda s: s.fillna(s.median())
)

print(df.isnull().sum())

### Step 2 — Handle outliers
**(a)** Drop impossible values (age > 100, employment > 60 yrs = data errors). **(b)** Cap extreme income with the IQR fence `Q3 + 1.5*IQR` — cap, don't delete.

In [ ]:
# (a) remove impossible values
df = df[df['person_age'] <= 100]
df = df[df['person_emp_length'] <= 60]

# (b) cap extreme income using the IQR fence
Q1 = df['person_income'].quantile(0.25)
Q3 = df['person_income'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR
df['person_income'] = df['person_income'].clip(upper=upper_fence)

df = df.reset_index(drop=True)
print('Rows:', df.shape[0], '| Max income:', df['person_income'].max())

### Step 3 — Create customer_id
Unique ID like `CUST_00001` as the first column.

In [ ]:
df.insert(0, 'customer_id', ['CUST_' + str(i + 1).zfill(5) for i in range(len(df))])
print(df.head())

### Save the clean file
Nulls + outliers handled, `customer_id` added, categories still as words.

In [ ]:
df.to_csv('./credit_risk_clean.csv', index=False)
print('Saved clean file:', df.shape)